In [9]:
import json
import os

# Define the default output path
default_output_path = "../data/2311_scopus_17416"

# Open and load the JSON file
with open(os.path.join(default_output_path, "result_llm_raw.json"), "r", encoding="utf-8") as file:
    data = json.load(file)

# Prepare an empty dictionary to hold results.
result = {}

# Iterate over every entry in the JSON data.
for entry in data:
    # Extract the file ID from the current entry.
    file_id = entry.get("File ID")
    
    # Get the list of institutions that contain ror IDs.
    institutions = entry.get("institutions_with_ror", [])
    
    # Build a list of ror_id values for the current entry, prepending the URL.
    # Only include the ror_id if it is not None and not the string "null".
    ror_ids = [
        "https://ror.org/" + inst.get("ror_id")
        for inst in institutions
        if "ror_id" in inst and inst.get("ror_id") not in [None, "null"]
    ]
    
    # Map the file ID to its corresponding list of updated ror ids.
    result[file_id] = ror_ids

# Define the output file path under the default directory
output_file_path = os.path.join(default_output_path, "result_llm.json")

# Save the result to a new JSON file with sorted keys
with open(output_file_path, "w", encoding="utf-8") as output_file:
    json.dump(result, output_file, ensure_ascii=False, indent=4, sort_keys=True)

print(f"Output saved to {output_file_path}")


Output saved to ../data/2311_scopus_17416/result_llm.json


In [10]:
import json
import re

# Paths to the input files
ground_truth_path = os.path.join(default_output_path, 'result_trie.json') 
result_llm_path = os.path.join(default_output_path,'result_llm.json')
output_path = os.path.join(default_output_path,'result_combined.json')

# Load JSON data from both files
with open(ground_truth_path, 'r', encoding='utf-8') as f:
    ground_truth = json.load(f)

with open(result_llm_path, 'r', encoding='utf-8') as f:
    result_llm = json.load(f)

# Prepare an output dictionary using keys from result_llm
combined = {}

for key in result_llm:
    # Start with the value from result_llm_v2
    value_llm = result_llm[key]
    
    # Check if the same key exists in ground_truth
    if key in ground_truth:
        value_ground = ground_truth[key]
        
        # Merge the two values based on their types:
        if isinstance(value_llm, list) and isinstance(value_ground, list):
            # Both are lists: concatenate them (optionally remove duplicates if needed)
            merged_value = value_llm + value_ground
        elif isinstance(value_llm, list):
            # LLM value is already a list, append ground truth value
            merged_value = value_llm + [value_ground]
        elif isinstance(value_ground, list):
            # Ground truth value is a list; put the LLM value at the beginning
            merged_value = [value_llm] + value_ground
        else:
            # Both are not lists: create a list with both values
            merged_value = [value_llm, value_ground]
    else:
        # If key does not exist in ground_truth, use the LLM value as-is
        merged_value = value_llm

    # Optionally, you can clean up the merged value (for example, using regex if needed)
    # merged_value = [re.sub(r'\s+', ' ', str(item)).strip() for item in merged_value] if isinstance(merged_value, list) else re.sub(r'\s+', ' ', str(merged_value)).strip()

    combined[key] = merged_value

# Write the combined result into a new JSON file
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(combined, f, indent=4)

print(f"Combined JSON file has been written to '{output_path}'")


Combined JSON file has been written to '../data/2311_scopus_17416/result_combined.json'


In [11]:
import json

# Load the result_llm.json to get the valid keys
with open(os.path.join(default_output_path, 'result_llm.json'), 'r') as f:
    result_llm_data = json.load(f)
    valid_keys = result_llm_data.keys()

# Load the groundTruth.json
with open(os.path.join(default_output_path, 'groundTruth.json'), 'r') as f:
    ground_truth_data = json.load(f)

# Filter the groundTruth data to only keep valid keys
filtered_ground_truth = {key: value for key, value in ground_truth_data.items() if key in valid_keys}

# Save the filtered ground truth to a new file
with open(os.path.join(default_output_path, 'groundTruth_clean.json'), 'w') as f:
    json.dump(filtered_ground_truth, f, indent=4)

print("Filtered ground truth has been saved to groundTruth_clean.json")


Filtered ground truth has been saved to groundTruth_clean.json
